In [ ]:
# =========================================
# Cell 1：挂载Google Drive
# =========================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================================
# Cell 2：进入项目目录 + 安装依赖 + 环境检查
# =========================================
%cd /content/drive/MyDrive/Wsy_DataBase/Capstone

!pip -q install torch torchvision tqdm pillow matplotlib

import os, torch
print("Capstone存在:", os.path.exists("/content/drive/MyDrive/Wsy_DataBase/Capstone"))
print("数据集存在:", os.path.exists("/content/drive/MyDrive/Wsy_DataBase/NEU_Seg-main_IMG"))
print("CUDA可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/drive/MyDrive/Wsy_DataBase/Capstone
Capstone存在: True
数据集存在: True
CUDA可用: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
# 1) 进入项目目录
%cd /content/drive/MyDrive/Wsy_DataBase/Capstone

# 2) 生成专门给Colab用的新脚本（不改原quick_run.py）
code = r'''
import argparse
import os
import random
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

from model.model import SelfNet
from model.unet import UNet


@dataclass(frozen=True)
class Sample:
    image_path: str
    mask_path: str


class NEUSegDataset(Dataset):
    def __init__(self, samples: List[Sample], image_transform=None):
        self.samples = samples
        self.image_transform = image_transform or transforms.ToTensor()

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        s = self.samples[idx]
        image = Image.open(s.image_path).convert("RGB")
        mask = Image.open(s.mask_path).convert("L")
        image_t = self.image_transform(image)
        mask_t = torch.from_numpy(np.array(mask, dtype=np.uint8)).long()
        return image_t, mask_t


def list_pairs(images_dir: str, masks_dir: str) -> List[Sample]:
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp"}
    image_files = [f for f in os.listdir(images_dir) if os.path.splitext(f.lower())[1] in valid_ext]
    samples = []
    for name in sorted(image_files):
        stem, _ = os.path.splitext(name)
        mp = os.path.join(masks_dir, f"{stem}.png")
        ip = os.path.join(images_dir, name)
        if os.path.isfile(mp):
            samples.append(Sample(ip, mp))
    return samples


def fast_hist(a, b, n):
    k = (a >= 0) & (a < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)


def iou_from_hist(hist):
    denom = hist.sum(1) + hist.sum(0) - np.diag(hist)
    return np.diag(hist) / np.maximum(denom, 1)


@torch.no_grad()
def evaluate_miou(model, loader, num_classes, device):
    model.eval()
    hist = np.zeros((num_classes, num_classes), dtype=np.float64)
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        preds = torch.argmax(model(images), dim=1)
        pn = preds.cpu().numpy().astype(np.int64)
        mn = masks.cpu().numpy().astype(np.int64)
        for p, m in zip(pn, mn):
            hist += fast_hist(m.flatten(), p.flatten(), num_classes)
    iou = iou_from_hist(hist)
    return iou, float(np.nanmean(iou))


@torch.no_grad()
def save_visual_examples(model, samples, out_dir, device, n):
    os.makedirs(out_dir, exist_ok=True)
    tfm = transforms.ToTensor()
    model.eval()
    for i, s in enumerate(samples[:n], start=1):
        img = Image.open(s.image_path).convert("RGB")
        x = tfm(img).unsqueeze(0).to(device)
        pred = torch.argmax(model(x), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        Image.fromarray(pred).save(os.path.join(out_dir, f"pred_{i:02d}.png"))
        Image.open(s.mask_path).convert("L").save(os.path.join(out_dir, f"gt_{i:02d}.png"))
        img.save(os.path.join(out_dir, f"img_{i:02d}.jpg"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["SelfNet", "UNet"], default="SelfNet")
    ap.add_argument("--num-classes", type=int, default=4)
    ap.add_argument("--train-n", type=int, default=1200)
    ap.add_argument("--val-n", type=int, default=300)
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--out-dir", default="quick_output_colab")
    args = ap.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    # 这里已经固定成你的Drive数据路径（无需再改原文件）
    base = "/content/drive/MyDrive/Wsy_DataBase/NEU_Seg-main_IMG"
    train_images = os.path.join(base, "images", "training")
    train_masks = os.path.join(base, "annotations", "training")
    if not (os.path.isdir(train_images) and os.path.isdir(train_masks)):
        raise SystemExit(f"数据路径不存在: {base}")

    all_samples = list_pairs(train_images, train_masks)
    if len(all_samples) < (args.train_n + args.val_n):
        raise SystemExit(f"样本不足: {len(all_samples)} < train_n+val_n={args.train_n + args.val_n}")

    random.shuffle(all_samples)
    train_samples = all_samples[:args.train_n]
    val_samples = all_samples[args.train_n:args.train_n + args.val_n]

    device = torch.device(args.device if (args.device != "cuda" or torch.cuda.is_available()) else "cpu")
    if args.model == "UNet":
        model = UNet(in_channels=3, num_classes=args.num_classes).to(device)
    else:
        model = SelfNet(in_channels=3, num_classes=args.num_classes).to(device)

    train_loader = DataLoader(NEUSegDataset(train_samples), batch_size=args.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(NEUSegDataset(val_samples), batch_size=args.batch_size, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    os.makedirs(args.out_dir, exist_ok=True)

    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for images, masks in tqdm(train_loader, desc=f"{args.model} epoch {epoch}/{args.epochs}"):
            images, masks = images.to(device), masks.to(device)
            loss = criterion(model(images), masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        iou, miou = evaluate_miou(model, val_loader, args.num_classes, device)
        print(f"[{args.model}][epoch {epoch}] loss={np.mean(losses):.4f} val_mIoU={miou:.4f} per_class={np.round(iou,4)}")

    ckpt = os.path.join(args.out_dir, f"{args.model}_quick.pth")
    torch.save(model.state_dict(), ckpt)
    save_visual_examples(model, val_samples, os.path.join(args.out_dir, "viz"), device, min(6, len(val_samples)))
    print("保存完成:", args.out_dir, "| ckpt:", ckpt)


if __name__ == "__main__":
    main()
'''
with open("quick_run_colab.py", "w", encoding="utf-8") as f:
    f.write(code)

print("已生成 /content/drive/MyDrive/Wsy_DataBase/Capstone/quick_run_colab.py")

/content/drive/MyDrive/Wsy_DataBase/Capstone
已生成 /content/drive/MyDrive/Wsy_DataBase/Capstone/quick_run_colab.py


In [ ]:
%cd /content/drive/MyDrive/Wsy_DataBase/Capstone

!python quick_run_colab.py --model UNet --epochs 12 --train-n 1200 --val-n 300 --batch-size 8 --device cuda --out-dir /content/drive/MyDrive/Wsy_DataBase/results_for_paper/unet_1200

/content/drive/MyDrive/Wsy_DataBase/Capstone
UNet epoch 1/12: 100% 150/150 [03:16<00:00,  1.31s/it]
[UNet][epoch 1] loss=0.4671 val_mIoU=0.3527 per_class=[0.9002 0.     0.3117 0.1988]
UNet epoch 2/12: 100% 150/150 [00:04<00:00, 35.97it/s]
[UNet][epoch 2] loss=0.2584 val_mIoU=0.4626 per_class=[0.8902 0.     0.6261 0.334 ]
UNet epoch 3/12: 100% 150/150 [00:04<00:00, 36.01it/s]
[UNet][epoch 3] loss=0.2089 val_mIoU=0.5799 per_class=[0.9231 0.0972 0.6635 0.6359]
UNet epoch 4/12: 100% 150/150 [00:04<00:00, 36.02it/s]
[UNet][epoch 4] loss=0.1839 val_mIoU=0.6407 per_class=[0.9265 0.3176 0.6765 0.6425]
UNet epoch 5/12: 100% 150/150 [00:04<00:00, 36.02it/s]
[UNet][epoch 5] loss=0.1682 val_mIoU=0.5548 per_class=[0.8943 0.2516 0.5082 0.565 ]
UNet epoch 6/12: 100% 150/150 [00:04<00:00, 35.98it/s]
[UNet][epoch 6] loss=0.1588 val_mIoU=0.5107 per_class=[0.9225 0.0354 0.6408 0.444 ]
UNet epoch 7/12: 100% 150/150 [00:04<00:00, 36.09it/s]
[UNet][epoch 7] loss=0.1523 val_mIoU=0.6676 per_class=[0.9394 0.39

In [ ]:
!python quick_run_colab.py --model SelfNet --epochs 12 --train-n 1200 --val-n 300 --batch-size 8 --device cuda --out-dir /content/drive/MyDrive/Wsy_DataBase/results_for_paper/ours_1200

SelfNet epoch 1/12: 100% 150/150 [00:02<00:00, 53.28it/s]
[SelfNet][epoch 1] loss=0.4464 val_mIoU=0.3674 per_class=[0.9035 0.     0.5602 0.0059]
SelfNet epoch 2/12: 100% 150/150 [00:02<00:00, 59.83it/s]
[SelfNet][epoch 2] loss=0.2588 val_mIoU=0.3900 per_class=[0.8596 0.     0.4621 0.2384]
SelfNet epoch 3/12: 100% 150/150 [00:02<00:00, 60.20it/s]
[SelfNet][epoch 3] loss=0.2239 val_mIoU=0.5230 per_class=[0.8814 0.2041 0.5846 0.4222]
SelfNet epoch 4/12: 100% 150/150 [00:02<00:00, 60.26it/s]
[SelfNet][epoch 4] loss=0.2045 val_mIoU=0.4056 per_class=[0.9126 0.0068 0.525  0.178 ]
SelfNet epoch 5/12: 100% 150/150 [00:02<00:00, 59.61it/s]
[SelfNet][epoch 5] loss=0.1875 val_mIoU=0.6020 per_class=[0.9329 0.3652 0.5839 0.5258]
SelfNet epoch 6/12: 100% 150/150 [00:02<00:00, 59.70it/s]
[SelfNet][epoch 6] loss=0.1797 val_mIoU=0.6498 per_class=[0.9349 0.3387 0.646  0.6797]
SelfNet epoch 7/12: 100% 150/150 [00:02<00:00, 59.99it/s]
[SelfNet][epoch 7] loss=0.1669 val_mIoU=0.5927 per_class=[0.9389 0.0997 

In [ ]:
!ls /content/drive/MyDrive/Wsy_DataBase/results_for_paper/unet_1200/viz
!ls /content/drive/MyDrive/Wsy_DataBase/results_for_paper/ours_1200/viz

gt_01.png  gt_04.png  img_01.jpg  img_04.jpg  pred_01.png  pred_04.png
gt_02.png  gt_05.png  img_02.jpg  img_05.jpg  pred_02.png  pred_05.png
gt_03.png  gt_06.png  img_03.jpg  img_06.jpg  pred_03.png  pred_06.png
gt_01.png  gt_04.png  img_01.jpg  img_04.jpg  pred_01.png  pred_04.png
gt_02.png  gt_05.png  img_02.jpg  img_05.jpg  pred_02.png  pred_05.png
gt_03.png  gt_06.png  img_03.jpg  img_06.jpg  pred_03.png  pred_06.png


In [ ]:
# =========================================
# 论文图片生成脚本（基于你已跑好的 unet_1200 / ours_1200）
# =========================================
import os
import glob
import numpy as np
from PIL import Image, ImageOps, ImageDraw

# 路径配置
ROOT = "/content/drive/MyDrive/Wsy_DataBase/results_for_paper"
UNET_VIZ = f"{ROOT}/unet_1200/viz"
OURS_VIZ = f"{ROOT}/ours_1200/viz"
OUT_FIG = f"{ROOT}/figures_final"
os.makedirs(OUT_FIG, exist_ok=True)

print("UNET_VIZ exists:", os.path.exists(UNET_VIZ))
print("OURS_VIZ exists:", os.path.exists(OURS_VIZ))

# ---------- 工具函数 ----------
def read_img(path):
    return Image.open(path).convert("RGB")

def read_mask(path):
    # mask可视化增强（防止看起来一片黑）
    m = Image.open(path).convert("L")
    m = ImageOps.autocontrast(m)
    return m.convert("RGB")

def add_title_bar(im, title):
    bar_h = 36
    canvas = Image.new("RGB", (im.width, im.height + bar_h), (30, 30, 30))
    canvas.paste(im, (0, bar_h))
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), title, fill=(255, 255, 255))
    return canvas

def make_row_panels(panels, spacing=8):
    h = max(p.height for p in panels)
    total_w = sum(p.width for p in panels) + spacing * (len(panels)-1)
    out = Image.new("RGB", (total_w, h), (255, 255, 255))
    x = 0
    for p in panels:
        out.paste(p, (x, 0))
        x += p.width + spacing
    return out

def save_with_header(row_img, header_text, save_path):
    header_h = 48
    head = Image.new("RGB", (row_img.width, header_h), (245, 245, 245))
    d = ImageDraw.Draw(head)
    d.text((12, 14), header_text, fill=(0, 0, 0))
    canvas = Image.new("RGB", (row_img.width, header_h + row_img.height), (255, 255, 255))
    canvas.paste(head, (0, 0))
    canvas.paste(row_img, (0, header_h))
    canvas.save(save_path)
    print("Saved:", save_path)

def mask_nonzero_ratio(mask_path):
    a = np.array(Image.open(mask_path).convert("L"))
    return float((a != 0).mean())

# ---------- 找可用样本编号 ----------
# 要求同时存在：ours的img/gt/pred + unet的pred
indices = []
for i in range(1, 200):
    idx = f"{i:02d}"
    req = [
        f"{OURS_VIZ}/img_{idx}.jpg",
        f"{OURS_VIZ}/gt_{idx}.png",
        f"{OURS_VIZ}/pred_{idx}.png",
        f"{UNET_VIZ}/pred_{idx}.png",
    ]
    if all(os.path.exists(p) for p in req):
        indices.append(idx)

if len(indices) == 0:
    raise RuntimeError("未找到可用样本，请先确认 unet_1200/viz 和 ours_1200/viz 已生成。")

print("可用样本编号:", indices)

# 选“更有缺陷内容”的样本优先展示
ranked = []
for idx in indices:
    gt_path = f"{OURS_VIZ}/gt_{idx}.png"
    score = mask_nonzero_ratio(gt_path)
    ranked.append((score, idx))
ranked.sort(reverse=True)

# 选前6个用于拼图（不够就按实际）
selected = [x[1] for x in ranked[:6]]
print("最终选用编号:", selected)

# 为不同图准备子集
sel3 = selected[:3] if len(selected) >= 3 else selected
sel4 = selected[:4] if len(selected) >= 4 else selected
sel2 = selected[:2] if len(selected) >= 2 else selected

# ---------- 图4-1：Baseline定性可视化 ----------
panels = []
for idx in sel3:
    img = read_img(f"{OURS_VIZ}/img_{idx}.jpg")
    gt = read_mask(f"{OURS_VIZ}/gt_{idx}.png")
    bp = read_mask(f"{UNET_VIZ}/pred_{idx}.png")
    panels.extend([
        add_title_bar(img, f"img_{idx}"),
        add_title_bar(gt,  f"gt_{idx}"),
        add_title_bar(bp,  f"baseline_pred_{idx}")
    ])
row = make_row_panels(panels)
save_with_header(row, "图4-1 Baseline定性可视化图（UNet）", f"{OUT_FIG}/图4-1_Baseline定性可视化图.png")

# ---------- 图4-2：Ours定性可视化 ----------
panels = []
for idx in sel3:
    img = read_img(f"{OURS_VIZ}/img_{idx}.jpg")
    gt = read_mask(f"{OURS_VIZ}/gt_{idx}.png")
    op = read_mask(f"{OURS_VIZ}/pred_{idx}.png")
    panels.extend([
        add_title_bar(img, f"img_{idx}"),
        add_title_bar(gt,  f"gt_{idx}"),
        add_title_bar(op,  f"ours_pred_{idx}")
    ])
row = make_row_panels(panels)
save_with_header(row, "图4-2 Ours定性可视化图（改进模型）", f"{OUT_FIG}/图4-2_Ours定性可视化图.png")

# ---------- 图6-1：四列对比（img|gt|unet|ours） ----------
panels = []
for idx in sel4:
    img = read_img(f"{OURS_VIZ}/img_{idx}.jpg")
    gt  = read_mask(f"{OURS_VIZ}/gt_{idx}.png")
    bp  = read_mask(f"{UNET_VIZ}/pred_{idx}.png")
    op  = read_mask(f"{OURS_VIZ}/pred_{idx}.png")
    panels.extend([
        add_title_bar(img, f"img_{idx}"),
        add_title_bar(gt,  f"gt_{idx}"),
        add_title_bar(bp,  f"unet_pred_{idx}"),
        add_title_bar(op,  f"ours_pred_{idx}")
    ])
row = make_row_panels(panels)
save_with_header(row, "图6-1 定性可视化对比图（img | gt | unet | ours）", f"{OUT_FIG}/图6-1_定性可视化对比图.png")

# ---------- 图6-2：误差案例分析（选2个缺陷较明显样本） ----------
# 这里简单定义“误差案例”：gt缺陷较多、且两模型预测非零占比差异较大
def pred_ratio(path):
    a = np.array(Image.open(path).convert("L"))
    return float((a != 0).mean())

err_rank = []
for idx in selected:
    gt_r = pred_ratio(f"{OURS_VIZ}/gt_{idx}.png")
    b_r  = pred_ratio(f"{UNET_VIZ}/pred_{idx}.png")
    o_r  = pred_ratio(f"{OURS_VIZ}/pred_{idx}.png")
    diff = abs(o_r - b_r)
    score = gt_r * 2 + diff * 3
    err_rank.append((score, idx, gt_r, b_r, o_r))
err_rank.sort(reverse=True)

use_err = err_rank[:2] if len(err_rank) >= 2 else err_rank
print("误差案例编号:", [x[1] for x in use_err])

panels = []
for _, idx, gt_r, b_r, o_r in use_err:
    img = read_img(f"{OURS_VIZ}/img_{idx}.jpg")
    gt  = read_mask(f"{OURS_VIZ}/gt_{idx}.png")
    bp  = read_mask(f"{UNET_VIZ}/pred_{idx}.png")
    op  = read_mask(f"{OURS_VIZ}/pred_{idx}.png")
    panels.extend([
        add_title_bar(img, f"img_{idx}"),
        add_title_bar(gt,  f"gt_{idx} nz={gt_r:.3f}"),
        add_title_bar(bp,  f"unet nz={b_r:.3f}"),
        add_title_bar(op,  f"ours nz={o_r:.3f}")
    ])

row = make_row_panels(panels)
save_with_header(row, "图6-2 误差案例分析图", f"{OUT_FIG}/图6-2_误差案例分析图.png")

print("\n全部生成完成，输出目录：", OUT_FIG)
print("输出文件：")
for f in sorted(glob.glob(f"{OUT_FIG}/*.png")):
    print("-", os.path.basename(f))

UNET_VIZ exists: True
OURS_VIZ exists: True
可用样本编号: ['01', '02', '03', '04', '05', '06']
最终选用编号: ['01', '03', '05', '04', '02', '06']
Saved: /content/drive/MyDrive/Wsy_DataBase/results_for_paper/figures_final/图4-1_Baseline定性可视化图.png
Saved: /content/drive/MyDrive/Wsy_DataBase/results_for_paper/figures_final/图4-2_Ours定性可视化图.png
Saved: /content/drive/MyDrive/Wsy_DataBase/results_for_paper/figures_final/图6-1_定性可视化对比图.png
误差案例编号: ['02', '03']
Saved: /content/drive/MyDrive/Wsy_DataBase/results_for_paper/figures_final/图6-2_误差案例分析图.png

全部生成完成，输出目录： /content/drive/MyDrive/Wsy_DataBase/results_for_paper/figures_final
输出文件：
- 图4-1_Baseline定性可视化图.png
- 图4-2_Ours定性可视化图.png
- 图6-1_定性可视化对比图.png
- 图6-2_误差案例分析图.png
